In [1]:
import os
import io
import joblib
import pandas as pd
import mysql.connector
import boto3
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Clase DataProcessor

Esta clase se encarga de todo el flujo de procesamiento de datos (alineado con el DAG, sin guardar en MinIO):

1. Cargar el dataset desde `covertype_raw`.
2. Limpiar datos faltantes o duplicados.
3. Dividir en train/val/test y preprocesar con el mismo pipeline que el DAG: numéricas (imputación mediana + StandardScaler), categóricas (imputación moda + OneHotEncoder). El preprocesador se ajusta solo con el train.
4. Devolver los conjuntos ya transformados.

In [2]:
# Mismas columnas que en el DAG (db_utils.preprocess_and_insert)
NUM_COLS = [
    "elevation", "aspect", "slope",
    "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways", "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]
CAT_COLS = ["wilderness_area", "soil_type"]


def _build_preprocessor():
    """Pipeline de preprocesamiento igual al del DAG: numéricas (imputer+scaler), categóricas (imputer+onehot)."""
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, NUM_COLS),
            ("cat", categorical_pipe, CAT_COLS),
        ],
        remainder="drop",
    )


class DataProcessor:
    """Carga desde covertype_raw, limpia y preprocesa como el DAG (sin guardar en MinIO). Ajusta el preprocesador solo con train."""

    def __init__(self):
        self.df = None
        self.preprocessor = None
        self.X_train = self.X_val = self.X_test = None
        self.y_train = self.y_val = self.y_test = None

    def load_data(self):
        conn = mysql.connector.connect(
            host=os.getenv("MYSQL_HOST", "mysql_db"),
            user=os.getenv("MYSQL_USER", "airflow"),
            password=os.getenv("MYSQL_PASSWORD", "airflow"),
            database=os.getenv("MYSQL_DATABASE", "covertype_data"),
        )
        self.df = pd.read_sql("SELECT * FROM covertype_raw", conn)
        conn.close()
        if "id" in self.df.columns:
            self.df = self.df.drop(columns=["id"])
        return self

    def clean_data(self):
        self.df = self.df.dropna()
        self.df = self.df.drop_duplicates()
        return self

    def extract_features_target(self, target_column="cover_type"):
        X = self.df.drop(columns=[target_column])
        y = self.df[target_column]
        return X, y

    def process(self, target_column="cover_type", test_size=0.3, val_size=0.5, random_state=42):
        """Carga, limpia, divide train/val/test y preprocesa (fit solo en train). Devuelve los 6 arrays."""
        self.load_data()
        self.clean_data()
        X, y = self.extract_features_target(target_column)
        X_train, X_val, X_test, y_train, y_val, y_test = split_data(
            X, y, test_size=test_size, val_size=val_size, random_state=random_state
        )
        self.preprocessor = _build_preprocessor()
        self.preprocessor.fit(X_train)
        self.X_train = self.preprocessor.transform(X_train)
        self.X_val = self.preprocessor.transform(X_val)
        self.X_test = self.preprocessor.transform(X_test)
        self.y_train = y_train.values
        self.y_val = y_val.values
        self.y_test = y_test.values
        return self.X_train, self.X_val, self.X_test, self.y_train, self.y_val, self.y_test

# División del Dataset

Esta función divide el dataset en tres conjuntos:

- **Train**: para entrenar el modelo.
- **Validation**: para ajustar hiperparámetros y evaluar durante entrenamiento.
- **Test**: para evaluación final del modelo.

Se usa un `random_state` para garantizar reproducibilidad.

In [3]:
def split_data(X, y, test_size=0.3, val_size=0.5, random_state=42):
    """Split data into train, validation, and test sets.
    
    Args:
        X: Features dataframe.
        y: Target variable.
        test_size (float): Proportion of data for testing.
        val_size (float): Proportion of temp data for validation.
        random_state (int): Random seed for reproducibility.
        
    Returns:
        tuple: (X_train, X_val, X_test, y_train, y_val, y_test)
    """
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=val_size, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

# Clase Model

Esta clase encapsula toda la lógica relacionada con el modelo:

- Construcción del modelo según el tipo seleccionado.
- Entrenamiento.
- Validación.
- Evaluación en test.
- Exportación a archivo `.pkl`.

Permite cambiar fácilmente entre distintos algoritmos sin modificar el resto del pipeline.

In [4]:
class Model:
    """Handles model building, training, validation, and export operations."""
    
    def __init__(self, model_type='svm', **model_params):
        """Initialize Model with model type and parameters.
        
        Args:
            model_type (str): Type of model to build ('svm', 'logistic_regression', 'random_forest').
            **model_params: Additional parameters for the model.
        """
        self.model_type = model_type
        self.model_params = model_params
        self.model = None
        self.X_train = None
        self.X_val = None
        self.X_test = None
        self.y_train = None
        self.y_val = None
        self.y_test = None
    
    def set_data(self, X_train, X_val, X_test, y_train, y_val, y_test):
        """Set training, validation, and test data.
        
        Args:
            X_train: Training features.
            X_val: Validation features.
            X_test: Test features.
            y_train: Training target.
            y_val: Validation target.
            y_test: Test target.
        """
        self.X_train = X_train
        self.X_val = X_val
        self.X_test = X_test
        self.y_train = y_train
        self.y_val = y_val
        self.y_test = y_test
        return self
    
    def build_model(self):
        """Build model based on model type and parameters."""
        if self.model_type == 'svm':
            self.model = SVC(**self.model_params)
        elif self.model_type == 'logistic_regression':
            self.model = LogisticRegression(**self.model_params)
        elif self.model_type == 'random_forest':
            self.model = RandomForestClassifier(**self.model_params)
        else:
            raise ValueError(f'Unsupported model type: {self.model_type}')
        return self
    
    def train(self):
        """Train the model on training data."""
        if self.model is None:
            raise ValueError('Model not built. Call build_model() first.')
        self.model.fit(self.X_train, self.y_train)
        return self
    
    def validate(self):
        """Validate the model on validation data.
        
        Returns:
            str: Classification report.
        """
        if self.model is None:
            raise ValueError('Model not trained. Call train() first.')
        predictions = self.model.predict(self.X_val)
        report = classification_report(self.y_val, predictions)
        return report
    
    def test(self):
        """Test the model on test data.
        
        Returns:
            str: Classification report.
        """
        if self.model is None:
            raise ValueError('Model not trained. Call train() first.')
        predictions = self.model.predict(self.X_test)
        report = classification_report(self.y_test, predictions)
        return report
    
    def export(self, file_path):
        """Export trained model to file.
        
        Args:
            file_path (str): Path to save the model file.
        """
        if self.model is None:
            raise ValueError('Model not trained. Call train() first.')
        
        # Validate if folder exists, if not create it
        folder = os.path.dirname(file_path)
        if not os.path.exists(folder):
            os.makedirs(folder)
        
        joblib.dump(self.model, file_path)
        return self

# Configuración del Entrenamiento

En este bloque se:

- Define la carpeta donde se guardarán los modelos.
- Se crea la carpeta si no existe.
- Se selecciona el tipo de modelo a entrenar.
- Se define el nombre con el que se guardará el modelo entrenado.

Esto permite controlar dinámicamente qué modelo se entrena y cómo se guarda.

In [5]:
model_type = "logistic_regression"
model_save_name = "logistic_regression_v2"

# Preprocesamiento de Datos

Aquí se ejecuta el pipeline de procesamiento:

1. Se instancia `DataProcessor`.
2. Se generan las variables X (features) y y (target).
3. Se divide el dataset en train, validation y test.
4. Se imprimen los tamaños de cada conjunto.

Este bloque solo se ejecuta una vez por entrenamiento.

In [6]:
print(f'{"="*60}')
print('DATA PREPROCESSING (igual al DAG: numéricas + categóricas, sin MinIO)')
print(f'{"="*60}')

data_processor = DataProcessor()
X_train, X_val, X_test, y_train, y_val, y_test = data_processor.process(target_column='cover_type')

print(f'Train samples: {len(X_train)}')
print(f'Validation samples: {len(X_val)}')
print(f'Test samples: {len(X_test)}')
print(f'Features: {X_train.shape[1]}')

DATA PREPROCESSING (igual al DAG: numéricas + categóricas, sin MinIO)


/tmp/ipykernel_114/1840916544.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  self.df = pd.read_sql("SELECT * FROM covertype_raw", conn)


Train samples: 27907
Validation samples: 5980
Test samples: 5981
Features: 34


# Configuración de Modelos Disponibles

Se define un diccionario con los modelos disponibles y sus hiperparámetros.

Luego se valida que el `model_type` seleccionado exista dentro de las opciones configuradas.

Esto permite flexibilidad para experimentar con distintos algoritmos.

In [7]:
model_configs = {
    'svm': {'kernel': 'rbf', 'C': 1.0},
    'logistic_regression': {'max_iter': 1000, 'random_state': 42},
    'random_forest': {'n_estimators': 100, 'random_state': 42}
}

if model_type not in model_configs:
    raise ValueError(f"Model type '{model_type}' is not supported")

model_params = model_configs[model_type]

# Entrenamiento y Validación

En este bloque:

1. Se instancia el modelo seleccionado.
2. Se cargan los datos de entrenamiento, validación y test.
3. Se construye el modelo.
4. Se entrena.
5. Se valida y se imprimen métricas.

Aquí ocurre el aprendizaje del modelo.

In [8]:
print(f'\n{"="*60}')
print(f'TRAINING {model_type.upper()} MODEL')
print(f'{"="*60}')

model = Model(model_type, **model_params)
model.set_data(X_train, X_val, X_test, y_train, y_val, y_test)

print('Building model...')
model.build_model()

print('Training model...')
model.train()

print('Validating model...')
validation_report = model.validate()
print(validation_report)


TRAINING LOGISTIC_REGRESSION MODEL
Building model...
Training model...
Validating model...
              precision    recall  f1-score   support

           1       0.86      0.91      0.88      3729
           2       0.79      0.71      0.75      1841
           3       0.74      0.67      0.70        30
           5       0.67      0.28      0.39        43
           6       0.92      0.90      0.91        51
           7       0.85      0.82      0.83       286

    accuracy                           0.84      5980
   macro avg       0.80      0.71      0.74      5980
weighted avg       0.83      0.84      0.83      5980



In [9]:
print(f'\n{"="*60}')
print('TEST RESULTS')
print(f'{"="*60}')

test_report = model.test()
print(test_report)

print(f'\n{"="*60}')
print('MODEL TRAINED, VALIDATED AND TESTED SUCCESSFULLY')
print(f'Saved in: {model_save_name}')
print(f'{"="*60}')


TEST RESULTS
              precision    recall  f1-score   support

           1       0.86      0.90      0.88      3717
           2       0.77      0.71      0.74      1808
           3       0.70      0.68      0.69        28
           5       0.85      0.20      0.33        54
           6       0.94      0.88      0.91        69
           7       0.84      0.83      0.83       305

    accuracy                           0.83      5981
   macro avg       0.83      0.70      0.73      5981
weighted avg       0.83      0.83      0.83      5981


MODEL TRAINED, VALIDATED AND TESTED SUCCESSFULLY
Saved in: logistic_regression_v2


# Guardado del modelo en MinIO

El módulo **MinIOModelSaver** se encarga de subir el modelo entrenado al almacenamiento de objetos MinIO (contenedor del proyecto), en lugar de usar un volumen compartido en disco.

- **Conexión:** usa las variables de entorno `MINIO_ENDPOINT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION` y opcionalmente `MINIO_BUCKET`.
- **Bucket:** por defecto `covertype-project`; los modelos se guardan bajo `models/` y los preprocesadores bajo `preprocessor/` con el mismo nombre (ej. `models/svm_v2.joblib` y `preprocessor/svm_v2.joblib`). Si el bucket no existe, se crea automáticamente.
- **Uso:** se instancia el saver, se sube el modelo y el preprocesador (mismo nombre). La API de inferencia descarga modelo y preprocesador por nombre al hacer predict.

In [10]:
class MinIOModelSaver:
    def __init__(self, bucket=None, prefix="v2/models/", preprocess_prefix="v2/preprocess/"):
        self.bucket = bucket or os.getenv("MINIO_BUCKET", "covertype-project")
        self.prefix = prefix.rstrip("/") + "/"
        self.preprocess_prefix = preprocess_prefix.rstrip("/") + "/"
        self._client = None

    def _get_client(self):
        if self._client is None:
            self._client = boto3.client(
                "s3",
                endpoint_url=os.getenv("MINIO_ENDPOINT"),
                aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
                aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
                region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1"),
            )
        return self._client

    def _ensure_bucket(self):
        client = self._get_client()
        buckets = [b["Name"] for b in client.list_buckets()["Buckets"]]
        if self.bucket not in buckets:
            client.create_bucket(Bucket=self.bucket)

    def save(self, model, name):
        self._ensure_bucket()
        buf = io.BytesIO()
        joblib.dump(model, buf)
        buf.seek(0)
        key = f"{self.prefix}{name}.joblib"
        self._get_client().upload_fileobj(buf, self.bucket, key)
        return f"s3://{self.bucket}/{key}"

    def save_preprocessor(self, preprocessor, name):
        self._ensure_bucket()
        buf = io.BytesIO()
        joblib.dump(preprocessor, buf)
        buf.seek(0)
        key = f"{self.preprocess_prefix}{name}.joblib"
        self._get_client().upload_fileobj(buf, self.bucket, key)
        return f"s3://{self.bucket}/{key}"

# Exportación del Modelo a MinIO

Se usa **MinIOModelSaver** para subir el modelo entrenado al contenedor MinIO (bucket y prefijo configurados en el módulo). La API de inferencia consume el modelo desde MinIO.

In [11]:
saver = MinIOModelSaver()
path = saver.save(model.model, model_save_name)
path_prep = saver.save_preprocessor(data_processor.preprocessor, model_save_name)
print(f'\nModel saved to MinIO: {path}')
print(f'Preprocessor saved to MinIO: {path_prep}')


Model saved to MinIO: s3://covertype-project/v2/models/logistic_regression_v2.joblib
Preprocessor saved to MinIO: s3://covertype-project/v2/preprocess/logistic_regression_v2.joblib
